In [7]:
# Clone the repository and create a symlink for the data
!git clone https://github.com/flyrank-bih/flyrank-ml-internship-starter.git
!ln -s flyrank-ml-internship-starter/data data

fatal: destination path 'flyrank-ml-internship-starter' already exists and is not an empty directory.


# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

> **Lane:** Lane 2 — Refresh / Content Opportunity Scoring
> **ML Task Type:** This is a **Ranking** problem solved via **Binary Classification**.
> We are not just trying to guess if a page is decaying; we are trying to output a probability score (from 0.0 to 1.0) so we can sort the entire inventory. The classifier’s probability output becomes our ranking mechanism, allowing us to build a strictly prioritized queue for human review.

In [8]:
import pandas as pd

# Load the starter dataset to verify the classification balance
df = pd.read_csv('data/raw/content_refresh_anonymized.csv')

# Check the distribution of the trend direction to see the class imbalance
class_counts = df['trend_direction'].value_counts(dropna=False)
class_percentages = df['trend_direction'].value_counts(normalize=True, dropna=False) * 100

summary_df = pd.DataFrame({
    'Count': class_counts,
    'Percentage (%)': class_percentages.round(1)
})
display(summary_df)

,Count,Percentage (%)
trend_direction,,
down,16262,54.2
stable,5962,19.9
up,4388,14.6
new,2236,7.5
flat,1152,3.8


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

> **Target Proxy:** Our proxy target for this slice is `is_declining_label`, defined as `trend_direction == "down"`.
> **Origin:** This is a **proxy label** derived from a current-window rule, not a pure future observed outcome. While a true capstone label would look at *future* 30-day traffic drops based on *past* 90-day features, this proxy is sufficient for our starter baseline to prove the pipeline works and to test our leakage controls.

In [9]:
# Define the proxy target as a binary label
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

# Show a sample of positive and negative cases using observable signals
columns_to_show = ['content_id', 'impressions_90d', 'avg_position', 'trend_direction', 'is_declining_label']
display(df[columns_to_show].sample(5, random_state=42))

,content_id,impressions_90d,avg_position,trend_direction,is_declining_label
2308,content_9824710082d8,283,21.3,stable,0
22404,content_3efa3a7c46bb,8878,8.8,up,0
23397,content_575dc8a2ab0f,3,0.0,new,0
25058,content_0dbd6911ba04,124,8.1,down,1
2664,content_bbaf87019afb,4294,30.9,down,1


## 3. Success metric

*One metric you can defend. What number means 'good'?*

> **Success Metric:** **Precision@K** (specifically, Precision@50).
> **Why it means 'good':** Overall accuracy or ROC-AUC is misleading in highly imbalanced triage datasets. The business team only has the capacity to manually review and update a limited number of pages (e.g., the top 50). Precision@50 measures strictly: out of the 50 pages the model told us to look at *first*, how many actually required attention? If Precision@50 is 0.74, it means 37 out of 50 recommendations were correct, minimizing wasted human effort.

In [10]:
# Demonstrate why top-K matters over total accuracy
total_declining = df['is_declining_label'].sum()
k_capacity = 50

print(f"Total pages flagged as declining by the proxy: {total_declining:,}")
print(f"Human review capacity (K): {k_capacity}")
print(f"We can only review {(k_capacity / total_declining) * 100:.2f}% of the declining pages.")
print("Conclusion: We must rank the top 50 perfectly. Accuracy on the bottom 29,000 pages does not matter to the reviewer.")

Total pages flagged as declining by the proxy: 16,262
Human review capacity (K): 50
We can only review 0.31% of the declining pages.
Conclusion: We must rank the top 50 perfectly. Accuracy on the bottom 29,000 pages does not matter to the reviewer.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

> **Unit of Analysis:** One row = **one pseudonymized content item (page)** per client at the time of the snapshot.
> The grain is flat across the current time window, meaning all temporal data (like 90-day impressions) has been aggregated into single column features for that specific page.

In [11]:
# Verify the grain: Is one row exactly one content item?
total_rows = len(df)
unique_content_ids = df['content_id'].nunique()

is_unique_grain = total_rows == unique_content_ids

print(f"Total rows: {total_rows}")
print(f"Unique content IDs: {unique_content_ids}")
print(f"Is the grain strictly one row per content item? {is_unique_grain}\n")

# Display the unit of analysis dataframe
display(df.head(3))

Total rows: 30000
Unique content IDs: 30000
Is the grain strictly one row per content item? True



,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,is_declining_label
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4,1
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7,1
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

> **Why ML beats a fixed rule:** Fixed rules (e.g., `IF impressions > 500 AND position > 10 THEN flag`) create massive "ties" where hundreds of pages receive the exact same priority.
> A fixed rule treats all flagged pages equally and ignores non-linear interactions—like how a 20% drop in traffic means something different for a 3-year-old page versus a 3-month-old page. A tree-based ML model captures these non-linear interactions and outputs a continuous probability, breaking ties and creating a strict 1-to-N ranking.

In [12]:
# Simulate a fixed rule to show the "tie" problem
# Rule: Page is older than 90 days, has > 500 impressions, and is declining
fixed_rule_flags = df[
    (df['content_age_days'] > 90) &
    (df['impressions_90d'] > 500) &
    (df['is_declining_label'] == 1)
]

num_tied_pages = len(fixed_rule_flags)

print(f"A basic fixed rule flags {num_tied_pages:,} pages.")
print(f"If we only have capacity for {k_capacity} reviews, the rule cannot tell us WHICH {k_capacity} to pick out of the {num_tied_pages:,}.")
print("An ML classifier solves this by sorting them by highest calculated probability of decline.")

A basic fixed rule flags 9,803 pages.
If we only have capacity for 50 reviews, the rule cannot tell us WHICH 50 to pick out of the 9,803.
An ML classifier solves this by sorting them by highest calculated probability of decline.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.